# 02 — Checking the data preparation (Section 2 of the paper)

Section 2 of Hamidieh (2018) describes three stages:

* **2.1 Element data** — building the table of 8 properties for 86 elements;
* **2.2 Superconducting material data** — cleaning the NIMS SuperCon download (steps 1–9, then removing
  elements with Z > 86 and repeated rows);
* **2.3 Feature extraction** — the 81 features.

**What this repo implements and what it checks.** Our code *uses* the author's final element table
(2.1) and *re-implements* the features (2.3, see notebook 01). The clean-up in 2.2 was done by the
author, mostly by hand in a spreadsheet (header of `reference/predict_tc/main_script_production_9.R`,
lines 37–73), before `unique_m.csv` was written. We start from that file, so for 2.2 we **check that
every step left the expected trace in the published data**; we do not re-run it.

The same checks run automatically in [`tests/test_data_preparation.py`](../tests/test_data_preparation.py)
(`uv run pytest`). The raw element table used here, [`data/reference/element_data_full.csv`](../data/reference/element_data_full.csv),
is the author's `element_data` object exported from `tc.RData` (before the adjustments of 2.1).

Legend: ✅ confirmed · ⚪ cannot be checked from the published files · ⚠️ differs from the paper's text.

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

from supercon import FEATURE_NAMES, featurize
from supercon.elements import ELEMENTS, PROPERTIES, load_element_table

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

unique_m = pd.read_csv(ROOT / "data" / "raw" / "unique_m.csv")
train = pd.read_csv(ROOT / "data" / "raw" / "train.csv")
raw = pd.read_csv(ROOT / "data" / "reference" / "element_data_full.csv", index_col=0).set_index("Element").loc[ELEMENTS]
used = load_element_table()
materials = set(unique_m["material"])

results = []

def check(section, claim, ok, observed):
    # ok: True -> confirmed, False -> differs, None -> cannot be checked
    mark = {True: "✅", False: "⚠️", None: "⚪"}[ok]
    results.append({"section": section, "paper says": claim, "": mark, "observed": observed})

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)

## 2.1 Element data

The paper: 86 elements, 46 variables from Mathematica `ElementData`, ~12% missing; 8 properties chosen;
La and Ce atomic radii replaced by their covalent radii (correlation ≈ 0.95); 1.5 added to every
electron affinity.

In [2]:
check("2.1", "86 elements (Z = 1..86)", raw["AtomicNumber"].tolist() == list(range(1, 87)), f"{len(raw)} elements")
check("2.1", "46 variables, ~12% missing", None,
      f"only {raw.shape[1]} columns were saved (10 dropped in R lines 357-367); {raw.isna().mean().mean():.1%} missing")
check("2.1", "8 properties (Table 1)", list(used.columns) == list(PROPERTIES), ", ".join(used.columns))

shift = (used["ElectronAffinity"] - raw["ElectronAffinity"]).round(10).unique()
check("2.1", "+1.5 added to every electron affinity", bool(np.allclose(shift, 1.5)), f"used - raw = {shift}")

r = raw[["AtomicRadius", "CovalentRadius"]].dropna().corr().iloc[0, 1]
check("2.1", "corr(atomic, covalent radius) ≈ 0.95", abs(r - 0.95) < 0.005, f"{r:.3f}")

pd.DataFrame({
    "raw AtomicRadius": raw.loc[["La", "Ce"], "AtomicRadius"],
    "raw CovalentRadius": raw.loc[["La", "Ce"], "CovalentRadius"],
    "AtomicRadius used": used.loc[["La", "Ce"], "AtomicRadius"],
})

,raw AtomicRadius,raw CovalentRadius,AtomicRadius used
Element,,,
La,NaN,207,195
Ce,NaN,204,185


The paper says La and Ce got their **covalent** radii, which in the author's own table are 207 and 204 pm.
The values actually used are **195 and 185 pm**; the R code (line 1061) says they come from
webelements.com. `train.csv` is consistent with 195/185 (notebook 01 reproduces it exactly), so the
paper's wording is inaccurate, not the data.

In [3]:
check("2.1", "La/Ce radius = covalent radius", False, "covalent = 207/204 pm, used = 195/185 pm (webelements); train.csv uses 195/185")

unchanged = all(
    np.array_equal(used[p].drop(["La", "Ce"]), raw[p].drop(["La", "Ce"]), equal_nan=True)
    for p in PROPERTIES if p != "ElectronAffinity"
)
check("2.1", "no other changes to the table", unchanged, "all other values identical to raw table")

## 2.2 SuperCon clean-up

Steps as numbered in the paper (pp. 5–6). Steps 1–3 and 6 (removing columns, sorting, fixing Tc values
shifted by one column for 25 `num` ids) act on columns and identifiers that are not in the published
files.

In [4]:
tc, m = unique_m["critical_temp"], unique_m["material"]

check("2.2", "1-3, 6: drop columns, sort, fix shifted Tc", None, "columns / num ids not published")

er = unique_m[m.str.startswith("Er1Ba2Cu3O7")]
check("2.2", "4: remove 3 records with Tc > 203 K",
      "La0.23Th0.77Pb3" not in materials and "Pb2C1Ag2O6" not in materials and tc.max() <= 203,
      f"2 formulas absent; Er1Ba2Cu3O7* max Tc {er.critical_temp.max()}; overall max {tc.max()} K ({m[tc.idxmax()]})")
check("2.2", "5: remove Tc = 0 or missing", bool(tc.notna().all() and (tc > 0).all()), f"min Tc = {tc.min()}")
check("2.2", "7: O7-X, O5+X ... -> O7, O5", not m.str.contains(r"[^A-Za-z0-9.]").any(),
      "no formula contains anything but letters, digits and '.'")

symbols = set(re.findall(r"[A-Z][a-z]?", " ".join(m)))
check("2.2", "8a: remove 'Yo...' (no such element)", symbols <= set(ELEMENTS), f"{len(symbols)} symbols, all valid")
n_symbols = m.map(lambda f: len(set(re.findall(r"[A-Z][a-z]?", f))))  # distinct elements written in the formula
n_nonzero = (unique_m[ELEMENTS] > 0).sum(axis=1)          # elements with a non-zero count
check("2.2", "8b: remove zero coefficients", bool((n_symbols == n_nonzero).all() and (unique_m[ELEMENTS] >= 0).all().all()),
      "every element written in a formula has a non-zero count "
      f"({(m.map(lambda f: len(re.findall(r'[A-Z][a-z]?', f))) > n_symbols).sum()} formulas repeat an element for different sites, e.g. Tl1Sr2Sr0.5Y0.5Cu2O7; counts are summed)")
check("2.2", "8c: remove Y2C2Br0.5!1.5", "Y2C2Br0.5!1.5" not in materials, "absent")
removed = ["Y1Ba2Cu3O6050", "Hg1234O10", "Nd185Ce0.15Cu1O4", "Bi1.6Pb0.4Sr2Cu3Ca2O1013", "Y1Ba2Cu285Ni0.15O7"]
fixed = ["Nd1.85Ce0.15Cu1O4", "Bi1.6Pb0.4Sr2Cu3Ca2O10.13", "Y1Ba2Cu2.85Ni0.15O7"]
check("2.2", "8d-h: specific removals / corrections",
      all(f not in materials for f in removed) and all(f in materials for f in fixed),
      "5 bad formulas absent, 3 corrected formulas present")
check("2.2", "9: columns 'critical_temp', 'material'", unique_m.columns[-2:].tolist() == ["critical_temp", "material"], "yes")
check("2.2", "31,611 rows -> 24,861 -> 23,888 (Z > 86)", None, "intermediate counts not published")
check("2.2", "remove repeated rows -> 21,263",
      len(unique_m) == 21263 and not unique_m.duplicated(ELEMENTS + ["critical_temp"]).any(),
      f"{len(unique_m)} rows, 0 repeated (counts, Tc)")

## 2.3 Features, and a cross-check with Section 3.1

The features are covered in notebook 01. The summary statistics printed in Section 3.1 (Tables 3 and 4,
Figure 3) give an independent check that the published data is the data the paper analysed.

In [5]:
X = featurize(unique_m[ELEMENTS])
err = np.abs(X.to_numpy() - train[FEATURE_NAMES].to_numpy()).max()
check("2.3", "81 features (Table 2)", err < 1e-9, f"all 21,263 x 81 values reproduced, max |diff| = {err:.1e}")

q = tc.quantile([0, 0.25, 0.5, 0.75, 1]).round(5).tolist()
check("3.1", "Tc min/Q1/med/Q3/max = 0.00021/5.4/20/63/185",
      bool(np.allclose(q, [0.00021, 5.4, 20, 63, 185], atol=0.05)), "/".join(f"{v:g}" for v in q))
check("3.1", "Tc mean/SD = 34.4/34.2", abs(tc.mean() - 34.4) < 0.05 and abs(tc.std() - 34.25) < 0.01,
      f"{tc.mean():.2f}/{tc.std():.2f} (paper rounds 34.25 down)")
iron, cuprate = (unique_m.Fe > 0).sum(), ((unique_m.Cu > 0) & (unique_m.O > 0)).sum()
check("3.1", "2,339 iron-based, 10,532 cuprates", (iron, cuprate) == (2339, 10532), f"{iron:,} / {cuprate:,}")
check("3.1", "O present in ~56% of materials", abs((unique_m.O > 0).mean() - 0.56) < 0.005, f"{(unique_m.O > 0).mean():.1%}")

## Summary of the checks

In [6]:
pd.DataFrame(results).style.hide(axis="index").set_properties(**{"text-align": "left"})

section,paper says,,observed
2.1,86 elements (Z = 1..86),✅,86 elements
2.1,"46 variables, ~12% missing",⚪,only 35 columns were saved (10 dropped in R lines 357-367); 9.1% missing
2.1,8 properties (Table 1),✅,"AtomicMass, FirstIonizationEnergy, AtomicRadius, Density, ElectronAffinity, FusionHeat, ThermalConductivity, Valence"
2.1,+1.5 added to every electron affinity,✅,used - raw = [1.5]
2.1,"corr(atomic, covalent radius) ≈ 0.95",✅,0.947
2.1,La/Ce radius = covalent radius,⚠️,"covalent = 207/204 pm, used = 195/185 pm (webelements); train.csv uses 195/185"
2.1,no other changes to the table,✅,all other values identical to raw table
2.2,"1-3, 6: drop columns, sort, fix shifted Tc",⚪,columns / num ids not published
2.2,4: remove 3 records with Tc > 203 K,✅,2 formulas absent; Er1Ba2Cu3O7* max Tc 93.6; overall max 185.0 K (H2S1)
2.2,5: remove Tc = 0 or missing,✅,min Tc = 0.00021


## Issues the original clean-up did not catch

These do not affect reproducing the paper (we use the data as published), but matter for feature
selection and for how we split train and test.

### 1. Misplaced decimal points

The paper fixes `Nd185Ce0.15Cu1O4` (should be Nd1.85…). Listing all formulas with a coefficient above
90 shows the same kind of error elsewhere, among many legitimate dilute alloys (e.g. `Mo99.99Co0.01`,
`La98Lu2`) and real compounds (`B105`, `Ba24Si100`):

In [7]:
big = unique_m[(unique_m[ELEMENTS] > 90).any(axis=1)]
print(len(big), "rows /", big.material.nunique(), "formulas with a coefficient > 90")
suspect = ["Pr185Ce0.15Cu1O4", "La1.85Sr0.15Cu98Ni0.02O4", "Rb3C60C60"]
unique_m.loc[m.isin(suspect), ["material", "critical_temp", "Pr", "Cu", "C"]]

44 rows / 43 formulas with a coefficient > 90


,material,critical_temp,Pr,Cu,C
2125,Rb3C60C60,29.00,0.0,0.0,120.0
4725,Pr185Ce0.15Cu1O4,20.00,185.0,1.0,0.0
17614,La1.85Sr0.15Cu98Ni0.02O4,21.96,0.0,98.0,0.0


* `Pr185Ce0.15Cu1O4` (Tc = 20 K) is almost certainly Pr$_{1.85}$Ce$_{0.15}$CuO$_4$, an electron-doped
  cuprate — the same slip as the Nd case the author corrected.
* `La1.85Sr0.15Cu98Ni0.02O4` should have Cu$_{0.98}$.
* `Rb3C60C60` is presumably Rb$_3$C$_{60}$ (Tc ≈ 29 K); written this way the carbon count doubles.

With 185 Pr atoms instead of 1.85, the weighted features of that material describe what is essentially
pure praseodymium.

### 2. The same composition written at different scales

Repeated rows were removed only when the element counts matched *exactly*. `Si1V3` and `Si0.25V0.75`
are the same material, and all 81 features (which depend only on proportions) are identical:

In [8]:
p = unique_m[ELEMENTS].div(unique_m[ELEMENTS].sum(axis=1), axis=0).round(10)
print(f"distinct formula strings        : {m.nunique():,}")
print(f"distinct normalized compositions: {len(p.drop_duplicates()):,}")
print(f"identical rows in train.csv     : {train.duplicated().sum()}")
(unique_m[m.isin(["Si1V3", "Si0.25V0.75"])]
 .groupby("material")["critical_temp"].agg(rows="count", min_Tc="min", max_Tc="max"))

distinct formula strings        : 15,542
distinct normalized compositions: 15,164
identical rows in train.csv     : 66


,rows,min_Tc,max_Tc
material,,,
Si0.25V0.75,7,16.6,17.1
Si1V3,12,15.9,17.1


**Consequence for modelling:** a train/test split grouped by formula string would still put `Si1V3` in
train and `Si0.25V0.75` in test. The grouped split should use the **normalized composition** (the
proportions $p_i$) as the group key.

### 3. A small inconsistency in the counts

The paper says the SuperCon download had 31,611 rows and that 6,750 were dropped for zero/missing Tc;
the author's R comments say 31,639 rows (line 28) and 31,639 − 24,861 = 6,778 dropped (line 133). Both
end at 24,861 rows, so it does not affect anything downstream.